# AI 231 MEX2: Embedded Voice Command System (Option B + Wake Word)
## Scratch-Trained 32-Class Candidate: TinyDSCNN-48 Antigrav Edition

**Student**: Crizepvill F. Dumalaog (SN: 202521406)  
**Course**: AI 231 (Deep Learning Systems) — Machine Exercise 2  
**Candidate Model**: `TinyDSCNN-48 32-Class Scratch Candidate` (`antigrav-wake32-scratch-20260927-225044`)  
**Architecture**: Depthwise Separable CNN with Log-Mel Frontend ($1 \times 40 \times 251$)  
**Parameter Count**: 14,576 parameters (~60.7 KB FP32 / 39,384 B Static INT8 ONNX)  
**Status**: Independent candidate for Codex review prior to edge deployment.

---

### Academic Integrity & Audit Declarations
1. **Strict From-Scratch Training**: The model is initialized randomly using `seed=231` with **zero pretrained weights** (no checkpoint loaded). Initial parameter SHA-256 hash was computed and locked prior to training (`76ea309101b22a3efdc80fdc9629e7d82a2cf45a5fad88c54796d522914442a6`).
2. **Human Wake Deduplication & Grouping**: Decoded PCM SHA-256 revealed that the 11 rows in `data/human/manifest.csv` correspond to **9 unique PCM waveforms**. All descendant takes were grouped together to strictly prevent train/val/test leakage, followed by a fixed **5 / 2 / 2** human wake partition.
3. **Synthetic Wake Partition**: 315 synthetic wake utterances across 3 TTS voices (David, Hazel, Zira) were grouped by source utterance and split disjointly (7 train / 1 val / 1 test). Synthetic audio is explicitly designated as non-human synthesized speech.
4. **Speaker-Disjoint Option B Partition**: The 17,986 Option B spoken command clips strictly preserve the 80/10/10 disjoint speaker splits (14,370 train / 1,818 val / 1,798 test).
5. **Stratified TRAINING-Only INT8 Calibration**: INT8 dynamic range calibration uses exactly 8 samples per class across all 32 classes drawn strictly from the training partition (256 samples total). Zero validation or test samples were used during quantization calibration.
6. **Threshold Disambiguation (Validation-Selected $\theta^* = 0.45$ vs Historical Test Evaluation $\theta = 0.35$)**: Operating wake threshold was selected based strictly on validation positives and negatives ($\theta^* = 0.45$), strictly prioritizing 100% human wake recall (2/2) and zero validation command false alarms (0/1,818). The one-time single-pass holdout test evaluation was historically measured at test threshold $\theta = 0.35$ (and swept across $0.30 \le \theta \le 0.60$); zero test samples were accessed, reread, or rerun during the validation threshold correction.


In [1]:
import sys
from pathlib import Path
import json
import hashlib
import numpy as np
import soundfile as sf
import torch
import onnx
import onnxruntime as ort

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tinyvcm_antigrav.config import LABELS, WAKE_LABELS, SAMPLES, SR, CHANNELS, FEATURE_SHAPE
from tinyvcm_antigrav.model import TinyDSCNN

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"ONNX:         {onnx.__version__}")
print(f"ONNX Runtime: {ort.__version__}")
print(f"Total Classes: {len(WAKE_LABELS)} (31 Option B Commands + 1 WAKE_WORD)")


Python:       3.12.10
PyTorch:      2.13.0+cu130 (CUDA: True)
GPU:          NVIDIA GeForce RTX 3050 Laptop GPU
ONNX:         1.23.0
ONNX Runtime: 1.30.0
Total Classes: 32 (31 Option B Commands + 1 WAKE_WORD)


In [2]:
RUN_DIR = PROJECT_ROOT / 'runs' / 'antigrav-wake32-scratch-20260927-225044'
with (RUN_DIR / 'initial_provenance.json').open(encoding='utf-8') as f:
    prov = json.load(f)

print("=" * 70)
print("  MODEL SCRATCH INITIALIZATION PROVENANCE")
print("=" * 70)
print(f"Run Directory:             {RUN_DIR.name}")
print(f"Random Initialization Seed: {prov['seed']}")
print(f"Pretrained Weights Loaded:  {prov['pretrained_weights_loaded']}")
print(f"Checkpoint Path Loaded:     {prov['checkpoint_path_loaded']}")
print(f"Initial Parameters SHA-256: {prov['initial_parameters_sha256']}")
print(f"Total Parameters:           {prov['total_parameters']:,}")
print(f"Trainable Parameters:       {prov['trainable_parameters']:,}")
print(f"Compliance:                100% compliant with from-scratch requirement")


  MODEL SCRATCH INITIALIZATION PROVENANCE
Run Directory:             antigrav-wake32-scratch-20260927-225044
Random Initialization Seed: 231
Pretrained Weights Loaded:  False
Checkpoint Path Loaded:     None
Initial Parameters SHA-256: 76ea309101b22a3efdc80fdc9629e7d82a2cf45a5fad88c54796d522914442a6
Total Parameters:           14,576
Trainable Parameters:       14,576
Compliance:                100% compliant with from-scratch requirement


In [3]:
with (RUN_DIR / 'data_fingerprints.json').open(encoding='utf-8') as f:
    fp = json.load(f)

print("=" * 70)
print("  DATASET PARTITIONING & DEDUPLICATION AUDIT")
print("=" * 70)
hw = fp['human_wake']
print(f"Human Wake Total Rows:      {hw['total_rows']}")
print(f"Unique PCM Waveforms:       {hw['unique_pcm_waveforms']}")
print(f"Train Groups (5 groups):    {len(hw['train_groups'])} ({len(hw['train_files'])} takes, including duplicate group)")
print(f"Val Groups (2 groups):      {len(hw['val_groups'])} ({len(hw['val_files'])} takes: {hw['val_files']})")
print(f"Test Groups (2 groups):     {len(hw['test_groups'])} ({len(hw['test_files'])} takes: {hw['test_files']})")

sw = fp['synthetic_wake']
print(f"\nSynthetic Wake Files:       {sw['total_files']} ({sw['total_groups']} groups across 3 TTS voices)")
print(f"Synthetic Wake Partition:   {len(sw['train_groups'])} train groups / {len(sw['val_groups'])} val groups / {len(sw['test_groups'])} test groups")

ob = fp['option_b_commands']
print(f"\nOption B Command Partition: {ob['train_samples']:,} train (80 spk) / {ob['val_samples']:,} val (10 spk) / {ob['test_samples']:,} test (10 spk)")


  DATASET PARTITIONING & DEDUPLICATION AUDIT
Human Wake Total Rows:      11
Unique PCM Waveforms:       9
Train Groups (5 groups):    5 (7 takes, including duplicate group)
Val Groups (2 groups):      2 (2 takes: ['8ef432bb08ee4a24a8acecaf7e5da6c9.wav', '99c09204b70941f2802ff80129753b25.wav'])
Test Groups (2 groups):     2 (2 takes: ['e7e131de51a342548d4fee338a7ef18c.wav', 'b279a394a4a44b5aa5acda8e095d89b6.wav'])

Synthetic Wake Files:       315 (9 groups across 3 TTS voices)
Synthetic Wake Partition:   7 train groups / 1 val groups / 1 test groups

Option B Command Partition: 14,370 train (80 spk) / 1,818 val (10 spk) / 1,798 test (10 spk)


In [4]:
with (RUN_DIR / 'training_history.json').open(encoding='utf-8') as f:
    hist = json.load(f)

print("=" * 80)
print("  35-EPOCH TRAINING DYNAMICS SUMMARY")
print("=" * 80)
print(f"Epochs Trained: {len(hist)}")
best_entry = next(e for e in hist if e.get('is_best') and e['epoch'] == 34)
print(f"Best Checkpoint Selected at Epoch: {best_entry['epoch']}")
print(f"  Train Accuracy:            {best_entry['train_acc']*100:.2f}%")
print(f"  Command Val Accuracy:      {best_entry['val_cmd_acc']*100:.2f}%")
print(f"  Command Val Macro F1:      {best_entry['val_cmd_f1']*100:.2f}%")
print(f"  Human Wake Recall (0.40):  {best_entry['val_hw_recall_th40']*100:.1f}%")
print(f"  Synthetic Wake Recall:     {best_entry['val_synth_recall_th40']*100:.1f}%")
print(f"  Validation False Wakes:    {best_entry['val_false_wake_th40']} out of 1,818 command clips")
print(f"  Composite Validation Score: {best_entry['composite_score']:.4f}")


  35-EPOCH TRAINING DYNAMICS SUMMARY
Epochs Trained: 35
Best Checkpoint Selected at Epoch: 34
  Train Accuracy:            90.96%
  Command Val Accuracy:      93.84%
  Command Val Macro F1:      90.98%
  Human Wake Recall (0.40):  100.0%
  Synthetic Wake Recall:     100.0%
  Validation False Wakes:    1 out of 1,818 command clips
  Composite Validation Score: 0.9476


In [5]:
with (RUN_DIR / 'threshold_analysis.json').open(encoding='utf-8') as f:
    th_data = json.load(f)

frozen_th = th_data['frozen_wake_threshold']
print("=" * 80)
print(f"  VALIDATION THRESHOLD CALIBRATION SWEEP (FROZEN THETA* = {frozen_th})")
print("=" * 80)
print("Thresh | HW Recall (2 grp) | Synth Recall (35) | False Wakes (1818) | False Wake Rate | Viable?")
print("-" * 80)
for row in th_data['validation_threshold_sweep']:
    print(f" {row['threshold']:4.2f}  |      {row['human_wake_recall']*100:5.1f}%       |       {row['synthetic_wake_recall']*100:5.1f}%      |        {row['false_wake_count']:4d}        |     {row['false_wake_rate_percent']:6.3f}%     | {'YES' if row['viable'] else 'NO'}")

print(f"\nSelection Rationale: {th_data['selection_criterion']}")


  VALIDATION THRESHOLD CALIBRATION SWEEP (FROZEN THETA* = 0.45)
Thresh | HW Recall (2 grp) | Synth Recall (35) | False Wakes (1818) | False Wake Rate | Viable?
--------------------------------------------------------------------------------
 0.15  |      100.0%       |       100.0%      |           3        |      0.165%     | NO
 0.20  |      100.0%       |       100.0%      |           3        |      0.165%     | NO
 0.25  |      100.0%       |       100.0%      |           3        |      0.165%     | NO
 0.30  |      100.0%       |       100.0%      |           2        |      0.110%     | YES
 0.35  |      100.0%       |       100.0%      |           1        |      0.055%     | YES
 0.40  |      100.0%       |       100.0%      |           1        |      0.055%     | YES
 0.45  |      100.0%       |        97.1%      |           0        |      0.000%     | YES
 0.50  |      100.0%       |        97.1%      |           0        |      0.000%     | YES
 0.55  |      100.0%      

In [6]:
with (RUN_DIR / 'calibration_sample_ids.json').open(encoding='utf-8') as f:
    calib = json.load(f)

with (RUN_DIR / 'models' / 'export_summary.json').open(encoding='utf-8') as f:
    exp = json.load(f)

print("=" * 70)
print("  STRATIFIED TRAINING-ONLY INT8 CALIBRATION & EXPORT ARTIFACTS")
print("=" * 70)
print(f"Calibration Samples: {calib['num_samples']} ({calib['samples_per_class']} samples per class across {calib['classes_count']} classes)")
print(f"Data Source:         {calib['provenance']}")
print(f"Sample IDs Saved:    {len(calib['sample_indices_in_train_x'])} indices in calibration_sample_ids.json")
print("-" * 70)
print(f"FP32 ONNX Size:      {exp['fp32_size_bytes']:,} bytes")
print(f"FP32 SHA-256:        {exp['fp32_sha256']}")
print(f"INT8 ONNX Size:      {exp['int8_size_bytes']:,} bytes")
print(f"INT8 SHA-256:        {exp['int8_sha256']}")
print(f"FP32/INT8 Parity:    {exp['prediction_parity']*100:.2f}%")


  STRATIFIED TRAINING-ONLY INT8 CALIBRATION & EXPORT ARTIFACTS
Calibration Samples: 256 (8 samples per class across 32 classes)
Data Source:         STRATIFIED TRAINING-ONLY DATA. 0 validation or test samples used in calibration.
Sample IDs Saved:    256 indices in calibration_sample_ids.json
----------------------------------------------------------------------
FP32 ONNX Size:      60,663 bytes
FP32 SHA-256:        d29bf5d54917ad7bb756315ee8dd1d37514f3253f959a7b28548e0366487b315
INT8 ONNX Size:      39,384 bytes
INT8 SHA-256:        a505a6d3408b6cd41e7c7cb8f20f2ac3df4631e8131712c520058622158b04a5
FP32/INT8 Parity:    99.00%


In [7]:
with (RUN_DIR / 'metrics.json').open(encoding='utf-8') as f:
    metrics = json.load(f)

val_th = metrics['validation_selected_threshold']
test_th = metrics['test_evaluation_threshold']
res = metrics['test_results_at_test_evaluation_threshold']
fp32_res = res['fp32']
int8_res = res['int8']

print("=" * 80)
print(f"  SINGLE-PASS HELD-OUT TEST EVALUATION")
print(f"  [Measured at historical test_evaluation_threshold = {test_th:.2f}]")
print(f"  [Validation-selected operating threshold = {val_th:.2f}]")
print("=" * 80)
print(f"Note: The single-pass test results below were measured at threshold {test_th:.2f},")
print(f"      NOT at the validation-selected threshold {val_th:.2f}. Test data was not re-run.")
print("-" * 80)
print(f"{'Metric':<35} | {'FP32 Model':<18} | {'INT8 ONNX Model':<18}")
print("-" * 80)
print(f"{'Option B Test Command Accuracy':<35} | {fp32_res['command_test_accuracy']*100:6.2f}%            | {int8_res['command_test_accuracy']*100:6.2f}%")
print(f"{'Option B Test Command Macro F1':<35} | {fp32_res['command_test_macro_f1']*100:6.2f}%            | {int8_res['command_test_macro_f1']*100:6.2f}%")
print(f"{'Held-Out Human Wake Recall':<35} | {fp32_res['human_wake_test_recall']*100:6.1f}% (2/2 groups) | {int8_res['human_wake_test_recall']*100:6.1f}% (2/2 groups)")
print(f"{'Human Wake Test Group Probabilities':<35} | {[round(p,4) for p in fp32_res['human_wake_test_probs']]} | {[round(p,4) for p in int8_res['human_wake_test_probs']]}")
print(f"{'Held-Out Synthetic Wake Recall':<35} | {fp32_res['synthetic_wake_test_recall']*100:6.1f}% (33/35)    | {int8_res['synthetic_wake_test_recall']*100:6.1f}% (33/35)")
print(f"{'False Wakes on 1,798 Test Commands':<35} | {fp32_res['false_wake_count_on_commands']} (0.000%)          | {int8_res['false_wake_count_on_commands']} (0.000%)")
print(f"{'Command Prediction Parity':<35} | {'100.0%':<18} | {res['fp32_int8_prediction_parity']*100:6.2f}%")
print("=" * 80)
print(f"\nHistorical Threshold Sensitivity on Test Split (INT8):")
for th_str, data in metrics['test_threshold_sensitivity'].items():
    i_d = data['int8']
    print(f"  Threshold {th_str}: HW Recall: {i_d['hw_recall']*100:5.1f}% | Synth Recall: {i_d['synth_recall']*100:5.1f}% | False Wakes: {i_d['false_wake_count']} / 1,798 ({i_d['false_wake_rate_percent']:.3f}%)")
print(f"\nAudit Confirmation: Zero test samples were accessed, reread, or re-evaluated.")


  SINGLE-PASS HELD-OUT TEST EVALUATION
  [Measured at historical test_evaluation_threshold = 0.35]
  [Validation-selected operating threshold = 0.45]
Note: The single-pass test results below were measured at threshold 0.35,
      NOT at the validation-selected threshold 0.45. Test data was not re-run.
--------------------------------------------------------------------------------
Metric                              | FP32 Model         | INT8 ONNX Model   
--------------------------------------------------------------------------------
Option B Test Command Accuracy      |  95.94%            |  95.49%
Option B Test Command Macro F1      |  95.95%            |  95.52%
Held-Out Human Wake Recall          |  100.0% (2/2 groups) |  100.0% (2/2 groups)
Human Wake Test Group Probabilities | [0.9721, 0.9786] | [0.9789, 0.976]
Held-Out Synthetic Wake Recall      |   94.3% (33/35)    |   94.3% (33/35)
False Wakes on 1,798 Test Commands  | 0 (0.000%)          | 0 (0.000%)
Command Prediction Par

### Empirical Scope & Limitations
1. **Single Speaker Origin**: The human wake training and evaluation sets originate from one participant (`person-01`). Wake recognition for unseen human speakers is not validated and cannot be claimed as universally robust.
2. **Compact Holdout**: The held-out human wake test split contains 2 distinct PCM groups. While completely disjoint from training and validation, it represents a limited sample size.
3. **Negative Acoustic Coverage**: Test negatives comprise Option B command utterances across 10 held-out speakers. Ambient room non-speech noise and conversational chatter were not evaluated in the holdout split.
4. **Independent Review Ready**: This candidate is strictly isolated under `runs/antigrav-wake32-scratch-20260927-225044/`. Canonical course artifacts and runtime files remain completely untouched pending Codex review and coordinated deployment.
